# NexaTel CDR Pipeline — Task 1: Ingest Raw CDR (Bronze)

> **Industry context — NexaTel Communications**  
> NexaTel is a mid-size telecom operator serving 5 million subscribers across Europe.  
> Every hour, network switches export **Call Detail Records (CDR)** to cloud storage.  
> This notebook is **Task 1** in the nightly Lakeflow Job. It validates the raw CDR  
> table is populated and sets a task value so downstream tasks know how many records  
> to expect.

**Lakeflow Job DAG position:** `ingest_cdr` → `cleanse_cdr` → `aggregate_usage`  
**Run-if condition on this task:** *(entry point — no upstream dependencies)*

In [ ]:
# ── CELL 1: Job parameters via widgets ───────────────────────────────────────
# Trainer: in a Lakeflow Job you pass parameters as job-level parameters.
# dbutils.widgets lets you read them OR test with defaults when running manually.

dbutils.widgets.text("billing_month", "2024-12", "Billing Month (YYYY-MM)")
dbutils.widgets.text("source_catalog", "trainer_demo", "Source Catalog")
dbutils.widgets.text("source_schema",  "demo_11",       "Source Schema")

billing_month  = dbutils.widgets.get("billing_month")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")

print(f"Parameters received:")
print(f"  billing_month  = {billing_month}")
print(f"  source_catalog = {source_catalog}")
print(f"  source_schema  = {source_schema}")

In [ ]:
# ── CELL 2: Read raw CDR and validate presence ────────────────────────────────
# Trainer: Bronze layer — we just confirm data is there; no transformations yet.
# Point out that we read the FULL table here; partitioning by billing_month
# would be applied once the table is partitioned in production.

from pyspark.sql import functions as F
import re

source_table = f"{source_catalog}.{source_schema}.raw_cdr"
print(f"Reading from: {source_table}")

raw_cdr = spark.read.table(source_table)

# Filter to the billing month supplied by the job parameter
year, month = billing_month.split("-")
monthly_cdr = raw_cdr.filter(
    (F.year("call_start_ts")  == int(year)) &
    (F.month("call_start_ts") == int(month))
)

total_records = monthly_cdr.count()
print(f"\nTotal CDR records for {billing_month}: {total_records:,}")

if total_records == 0:
    dbutils.notebook.exit(f"ERROR: No CDR records found for {billing_month}")

print("\nSample records (Bronze — raw, untransformed):")
monthly_cdr.show(5, truncate=False)

In [ ]:
# ── CELL 3: Quick schema and null overview ────────────────────────────────────
# Trainer: show the schema and highlight columns that will need cleaning.
# Intentionally revealed: NULL subscriber_id, negative duration_seconds.

print("=== Schema ===")
monthly_cdr.printSchema()

print("\n=== Null counts per column ===")
null_counts = [
    (col, monthly_cdr.filter(F.col(col).isNull()).count())
    for col in monthly_cdr.columns
]
for col, cnt in null_counts:
    bar = "⚠️ " if cnt > 0 else "  "
    print(f"  {bar}{col:<25} {cnt:>6,} nulls")

negative_dur = monthly_cdr.filter(F.col("duration_seconds") < 0).count()
invalid_ids  = monthly_cdr.filter(
    F.col("subscriber_id").isNotNull() &
    ~F.col("subscriber_id").rlike(r"^SUB-\d{5}$")
).count()
print(f"\n  ⚠️ {'negative duration_seconds':<25} {negative_dur:>6,} records")
print(f"  ⚠️ {'invalid subscriber_id format':<25} {invalid_ids:>6,} records")

In [ ]:
# ── CELL 4: Pass record count to downstream tasks via taskValues ──────────────
# Trainer: dbutils.jobs.taskValues lets tasks share data without writing a table.
# Task 3 (aggregate_usage) can read this value to do a sanity-check reconciliation.
#
# In a real Lakeflow Job this would store the value in the job run context.
# When running this notebook standalone the call is silently ignored.

call_type_counts = {
    row["call_type"]: row["count"]
    for row in monthly_cdr.groupBy("call_type").count().collect()
}

try:
    dbutils.jobs.taskValues.set(key="total_raw_records",   value=total_records)
    dbutils.jobs.taskValues.set(key="call_type_counts",    value=str(call_type_counts))
    dbutils.jobs.taskValues.set(key="billing_month",       value=billing_month)
    print(f"Task values set:")
    print(f"  total_raw_records = {total_records:,}")
    print(f"  call_type_counts  = {call_type_counts}")
    print(f"  billing_month     = {billing_month}")
except Exception:
    # Running outside a Lakeflow Job — taskValues not available
    print(f"[Standalone mode] Would set task values:")
    print(f"  total_raw_records = {total_records:,}")
    print(f"  call_type_counts  = {call_type_counts}")
    print(f"  billing_month     = {billing_month}")

print("\n✓ Task 1 (ingest_cdr) complete — downstream tasks may start.")
dbutils.notebook.exit("SUCCESS")